# LegalEase AI — Courtroom NER Fine-Tuning (LHC + SCP)

Fine-tunes a pretrained multilingual transformer for token classification
(Named Entity Recognition) on Pakistani courtroom judgment text — Lahore
High Court (**LHC**) and Supreme Court of Pakistan (**SCP**) — to recognise
entities like person names, organisations, dates, case numbers, and money
amounts inside judgment text.

**Run this notebook in Google Colab** (Runtime → Change runtime type → GPU
recommended, though it'll run on CPU too, just slower).

**Before you start**: upload `ner_courtroom_data.zip`
(`data/raw/ner_courtroom_data.zip` in the project repo) to this Colab
session's file storage — the cell below unzips it. It contains:

```
ner_courtroom/LHC/train.txt
ner_courtroom/LHC/valid.txt
ner_courtroom/SCP/train.txt
ner_courtroom/SCP/valid.txt
ner_courtroom/SCP/test.txt
```

Note **LHC has no `test.txt`** — only `train.txt` and `valid.txt` exist for
that source. The evaluation section later handles this explicitly rather
than pretending a test split exists where it doesn't.


## 1. Install dependencies

`transformers` + `datasets` for the model/data pipeline, `seqeval` for
entity-level precision/recall/F1 (it understands B-/I- tagging and scores
whole entities, not individual tokens), `evaluate` as the metric-computation
wrapper `Trainer` expects, `accelerate` for `Trainer` GPU/mixed-precision
support.


In [ ]:
!pip install -q transformers datasets seqeval evaluate accelerate


## 2. Upload and unzip the data

Run this cell, then use the file picker to upload `ner_courtroom_data.zip`
from your machine (it's at `data/raw/ner_courtroom_data.zip` in the
project repo).


In [ ]:
from google.colab import files

uploaded = files.upload()  # pick ner_courtroom_data.zip in the dialog
zip_name = next(iter(uploaded))
print(f"Uploaded: {zip_name}")


In [ ]:
import zipfile
from pathlib import Path

DATA_DIR = Path("ner_courtroom_data")
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(DATA_DIR)

# Should show LHC/ and SCP/ subfolders, each with train/valid(/test).txt
for p in sorted(DATA_DIR.rglob("*.txt")):
    print(p, "-", p.stat().st_size, "bytes")


## 3. Parse the CoNLL-format files

Both sources are one token + tag per line, blank line = sentence boundary —
standard CoNLL format. **LHC uses tab-separated columns, SCP uses
space-separated columns**; `line.split()` (no argument) splits on *any*
whitespace run, so one parser handles both without special-casing.

The raw files aren't perfectly clean — a handful of lines don't split into
exactly two fields (e.g. two lines that got concatenated without a
newline during data collection). The parser below skips those and reports
how many, rather than crashing or silently mis-parsing them.


In [ ]:
from pathlib import Path


def parse_conll(path: Path):
    """Yields (tokens, tags) per sentence. Returns (sentences, n_skipped)."""
    sentences = []
    tokens, tags = [], []
    n_skipped = 0

    with open(path, encoding="utf-8", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                if tokens:
                    sentences.append((tokens, tags))
                tokens, tags = [], []
                continue

            parts = line.split()
            if len(parts) != 2:
                # Malformed line (e.g. two rows concatenated in the source
                # file) — skip it rather than guess which part is the tag.
                n_skipped += 1
                continue

            token, tag = parts
            tokens.append(token)
            tags.append(tag)

    if tokens:  # file may not end with a trailing blank line
        sentences.append((tokens, tags))

    return sentences, n_skipped


def load_split(path: Path, label: str):
    sentences, n_skipped = parse_conll(path)
    print(f"{label:22s} {len(sentences):6d} sentences  ({n_skipped} malformed lines skipped)")
    return sentences


In [ ]:
lhc_train = load_split(DATA_DIR / "LHC" / "train.txt", "LHC train")
lhc_valid = load_split(DATA_DIR / "LHC" / "valid.txt", "LHC valid")

scp_train = load_split(DATA_DIR / "SCP" / "train.txt", "SCP train")
scp_valid = load_split(DATA_DIR / "SCP" / "valid.txt", "SCP valid")
scp_test = load_split(DATA_DIR / "SCP" / "test.txt", "SCP test")

# LHC ships no test.txt — don't fabricate one. Evaluation further down
# reports the combined valid set and, separately, the SCP-only test set.


## 4. Combine LHC + SCP into one training set

Concatenating the sentence lists is sufficient — no re-indexing or
alignment needed since each sentence already carries its own tokens/tags.


In [ ]:
train_sentences = lhc_train + scp_train
valid_sentences = lhc_valid + scp_valid

print(f"Combined training set:   {len(train_sentences)} sentences "
      f"({len(lhc_train)} LHC + {len(scp_train)} SCP)")
print(f"Combined validation set: {len(valid_sentences)} sentences "
      f"({len(lhc_valid)} LHC + {len(scp_valid)} SCP)")
print(f"SCP-only test set:       {len(scp_test)} sentences (LHC has no test split)")


## 5. Build the label list

Collect every unique tag seen across all loaded splits (train + valid +
test), so the model's output layer covers every label that could appear at
evaluation time too.

**Data-quality note**: LHC and SCP don't use identical tag spelling for the
same concept — e.g. LHC has `B-caseNo.` (capital N, trailing period) where
SCP has `B-caseno` (lowercase, no period), and similarly `refCourt` vs
`refcourt`. This notebook treats them as **distinct labels**, exactly as
they appear in the raw data — no normalisation was requested. If you want
LHC and SCP's case-number/reference tags merged into one label, that's a
deliberate follow-up decision, not something to silently do here.


In [ ]:
all_tags = set()
for _, tags in train_sentences + valid_sentences + scp_test:
    all_tags.update(tags)

# "O" first, then everything else alphabetically — just for readable output;
# the actual id assignment is what matters to the model, not the ordering.
label_list = ["O"] + sorted(t for t in all_tags if t != "O")
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"{len(label_list)} unique labels:")
for l in label_list:
    print(" ", l)


## 6. Build Hugging Face `Dataset` objects


In [ ]:
from datasets import Dataset


def to_hf_dataset(sentences):
    return Dataset.from_dict({
        "tokens": [s[0] for s in sentences],
        "tags": [[label2id[t] for t in s[1]] for s in sentences],
    })


train_ds = to_hf_dataset(train_sentences)
valid_ds = to_hf_dataset(valid_sentences)
scp_test_ds = to_hf_dataset(scp_test)

train_ds, valid_ds, scp_test_ds


## 7. Tokenize and align labels to subwords

`distilbert-base-multilingual-cased` is a subword tokenizer — a single
word like `"Ghulam"` can split into multiple subword tokens. Each subword
needs a label, but we only want to score the *first* subword of each
original word (standard practice for token classification); the rest get
label `-100`, which `Trainer`'s loss function is set up to ignore.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_and_align_labels(batch):
    tokenized = tokenizer(
        batch["tokens"],
        truncation=True,
        max_length=256,
        is_split_into_words=True,
    )

    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # special token ([CLS], [SEP], padding)
            elif word_id != prev_word_id:
                label_ids.append(tags[word_id])  # first subword of a word
            else:
                label_ids.append(-100)  # subsequent subwords of the same word
            prev_word_id = word_id
        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


train_tok = train_ds.map(tokenize_and_align_labels, batched=True, remove_columns=train_ds.column_names)
valid_tok = valid_ds.map(tokenize_and_align_labels, batched=True, remove_columns=valid_ds.column_names)
scp_test_tok = scp_test_ds.map(tokenize_and_align_labels, batched=True, remove_columns=scp_test_ds.column_names)


## 8. Load the pretrained model

`AutoModelForTokenClassification` adds a fresh linear classification head
on top of the pretrained transformer, sized to our label list — the
transformer's pretrained weights get fine-tuned, the head is trained from
scratch.


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)


## 9. Metrics — entity-level precision/recall/F1 via `seqeval`

`seqeval` understands B-/I- tagging: it scores whole entities (e.g. a
3-token person name counts as one correct/incorrect prediction, not three),
and its `classification_report`-style output gives **per-entity-type**
precision/recall/F1 (person, org, date, caseNo, money, ...) plus overall
micro/macro averages — exactly what's needed here.


In [ ]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [id2label[l] for l in label_row if l != -100]
        for label_row in labels
    ]
    true_predictions = [
        [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    # Flatten seqeval's per-entity-type dict into a single results dict so
    # Trainer's log output shows precision/recall/f1 per entity type, e.g.
    # "per_org_f1", "per_date_f1", "overall_f1".
    flat = {}
    for key, val in results.items():
        if isinstance(val, dict):
            for sub_key, sub_val in val.items():
                flat[f"{key}_{sub_key}"] = sub_val
        else:
            flat[key] = val
    return flat


## 10. Fine-tune

Standard `Trainer` setup. Adjust `num_train_epochs`/`per_device_train_batch_size`
for your Colab GPU tier if you hit an out-of-memory error (T4 free tier:
batch size 8-16 is usually safe at `max_length=256`).


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="ner_model_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="overall_f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


## 11. Evaluate

Two evaluations, reported separately since they cover different data:

1. **Combined LHC+SCP validation set** — held out during training, same
   mix the model was tuned on.
2. **SCP-only test set** — a second, unseen holdout, but only from SCP
   since LHC provides no test split at all. Don't read this as "the"
   test-set number for LHC performance — it isn't one.


In [ ]:
print("=" * 60)
print("Combined LHC+SCP validation set")
print("=" * 60)
valid_metrics = trainer.evaluate(eval_dataset=valid_tok)
for k, v in sorted(valid_metrics.items()):
    print(f"  {k}: {v}")


In [ ]:
print("=" * 60)
print("SCP-only test set (LHC has no test split)")
print("=" * 60)
test_metrics = trainer.evaluate(eval_dataset=scp_test_tok)
for k, v in sorted(test_metrics.items()):
    print(f"  {k}: {v}")


For a cleaner per-entity-type table (rather than the flattened
`Trainer` metric dict above), run `seqeval` directly:


In [ ]:
from seqeval.metrics import classification_report

predictions, labels, _ = trainer.predict(scp_test_tok)
predictions = np.argmax(predictions, axis=2)

true_labels = [[id2label[l] for l in row if l != -100] for row in labels]
true_predictions = [
    [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
    for pred_row, label_row in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions, digits=3))


## 12. Save the fine-tuned model

Saves the model + tokenizer to a folder, zips it, and offers it as a
Colab download. Grab the zip before the Colab runtime disconnects —
`ner_model_checkpoints/` and `ner_model_output/` don't persist across
sessions.


In [ ]:
OUTPUT_DIR = "ner_model_output"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# label maps aren't automatically reloadable from config in every HF
# version — save them explicitly too, so a fresh load can rebuild them.
import json as _json
with open(f"{OUTPUT_DIR}/label_list.json", "w") as f:
    _json.dump(label_list, f, indent=2)

print(f"Saved model + tokenizer + label_list.json to ./{OUTPUT_DIR}/")


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
files.download(f"{OUTPUT_DIR}.zip")
